# RAG (Retrieval‑Augmented Generation)

`retrieval` means the process of finding and selecting the most relevant pieces of information from an external knowledge base to support the LLM’s answer.

Think of retrieval as the search engine inside RAG. The LLM is the “writer.” Retrieval is the “librarian” who hands the writer the right book pages to answer your question.

![my understanding](image-12.png)

Some things not to love about RAG
- It’s a hack
- “It’s empirical” aka trial & error
- Unexpected ‘I don’t know’
- Great idea can have poor results

# Measuring Performance and Iterating on Your Pipeline

## Why RAG?

### 1. Save on context length

#### Why Context Length Matters ?

LLMs have a fixed context window (e.g., 4k, 8k, 32k tokens). If you feed too much raw text (like entire documents), you quickly hit that limit. Once exceeded, the model can’t “see” the rest, leading to incomplete or wrong answers.

RAG avoids stuffing the entire knowledge base into the prompt. Instead, it:
- `Chunks documents` → Splits large texts into smaller, manageable pieces.
- `Embeds chunks into vectors` → Each chunk is represented in a vector space.
- `Retrieves only relevant chunks` → At query time, the retriever fetches just the top‑k most relevant pieces.
- `Augments the prompt` → Only those selected chunks are injected into the LLM’s context window.

So we ccan say instead of wasting tokens on irrelevant text, the model only sees what’s necessary to answer the question.

Imagine a 200‑page manual: Without RAG: You’d try to feed the whole manual → impossible, exceeds context length. With RAG: You embed all pages, then for a query like “How to reset the device?”, you retrieve only the 2–3 chunks about resetting. Result: The LLM uses maybe 1,000 tokens instead of 200,000.

![RAG saves on context length](image-8.png)

### 2. How RAG given Give an LLM vast expertise?

- **Limits of a Standalone LLM**
    - A pretrained LLM knows a lot, but only up to its training cutoff date.
    - It cannot access private, domain‑specific, or constantly changing knowledge (e.g., medical guidelines, company manuals, legal documents).
    - Expanding its expertise by retraining is expensive and slow.

- **How RAG Expands Expertise**:
    - RAG acts like a knowledge plug‑in:
    - External Knowledge Base → You load domain documents (manuals, papers, FAQs, datasets).
    - Vectorization → These documents are chunked and embedded into a searchable vector store.
    - Retrieval → When asked a question, the system fetches the most relevant chunks.
    - Augmentation → Those chunks are injected into the LLM’s prompt.
    - Generation → The LLM produces an answer that combines its general reasoning ability with the retrieved expert knowledge.

- The LLM doesn’t need to “know everything” itself. Instead, it becomes an expert assistant by pulling in the right context at the right time.

Analogy:

Think of the LLM as a brilliant student: Alone, it knows a lot but forgets details. With RAG, it has a library card: it can instantly pull the right book off the shelf and answer with authority.

RAG provided Domain expertise, Up‑to‑date knowledge, Scalable expertise.

### 3. Avoid distracting with irrelevant context

## Beware about RAG:

### 1. Why It’s Considered a Hack ?

`Workaround, not redesign` → Instead of retraining the LLM with new knowledge, RAG bolts on a retrieval system that feeds context at runtime. It’s clever, but it’s not a fundamental fix to the model’s knowledge gap.

`Empirical tuning` → Success depends heavily on trial‑and‑error choices like chunk size, embedding model, retrieval strategy, and top‑k selection. There’s no universal formula; you experiment until it “works well enough.”

`Fragile behavior` → Small changes in chunking or retrieval can cause big swings in answer quality. That unpredictability makes it feel like a patch rather than a robust solution.

`“I don’t know” problem` → If retrieval fails or returns irrelevant chunks, the LLM may produce weak or nonsensical answers. The system’s reliability hinges on retrieval quality.

![LLM + RAG “Hack”](image-9.png)

On the right: the LLM patched with a retrieval system, pulling relevant chunks from a vector database to inject context. This makes clear why people call RAG a “hack”: it’s a clever bolt‑on workaround that patches the gap, but relies on empirical tuning and context injection rather than fundamentally expanding the model’s training.

### 2.  “It’s empirical” aka trial & error

What “Empirical” Means Here ?

In RAG, empirical means you discover what works by experimentation, not by a fixed theory or formula.
There’s no universal rule that guarantees optimal performance — you tweak parameters, observe results, and iterate.

![Where Trial & Error Appears](image-10.png)

So RAG’s “empirical” nature isn’t a flaw — it’s a research opportunity. By systematically experimenting with chunking, embeddings, and evaluation metrics, you transform the “hack” into a science of retrieval optimization.

### 3. “Unexpected ‘I don’t know’ problem”

The “Unexpected ‘I don’t know’ problem” in RAG happens when the retrieval stage fails to supply the LLM with the right context — so the model, even though capable of reasoning, ends up with nothing useful to reason about.

How It Arises ?

- `Retrieval Misses the Target`: The query embedding doesn’t match any relevant document vectors closely enough. The retriever returns chunks that look similar semantically but don’t actually contain the needed facts.
- `Poor Chunking or Indexing`: If documents were chunked too coarsely or too finely, the right information may be split or buried. The retriever can’t find the precise passage, so the LLM sees irrelevant or incomplete text.
- `Prompt Context Too Sparse`: Even if retrieval succeeds, the injected text might be too short or vague. The LLM then says “I don’t know” (or hallucinates) because the prompt lacks grounding.
- `Embedding Drift Across Domains`: Embeddings trained on general text may misinterpret domain‑specific queries (e.g., medical or legal terms). The retriever thinks it found matches, but they’re semantically off.

Why It’s “Unexpected”

The system looks fine — embeddings, vector store, LLM all working — yet the answer fails. It’s not a bug in the model; it’s a retrieval‑quality issue. That’s why RAG evaluation (EVALS) is crucial: you measure retrieval precision and answer faithfulness to catch these silent failures.

How to Reduce It?

- Improve chunking strategy (semantic or hierarchical).
- Use domain‑specific embeddings.
- Tune top‑k retrieval and similarity thresholds.
- Add fallback logic (e.g., “no relevant context found” → trigger broader search).

![flow diagram](image-11.png)

### 4. Great idea can have poor results

The phrase “Great idea can have poor results” in RAG points to the gap between the conceptual brilliance of Retrieval‑Augmented Generation and the messy reality of implementation.

# Retrieval Metrics, LLM as Judge, and Golden Data

#### Retrieval Metrics:

These measure how well your retriever finds the right chunks. 
- `MRR (Mean Reciprocal Rank)` → Did the first relevant chunk appear near the top?
- `nDCG` (Normalized Discounted Cumulative Gain) → Were relevant chunks ranked higher than irrelevant ones?
- `Recall@K` → Out of all test queries, how often did the relevant chunk appear in the top K results?
- `Precision@K` → Of the top K chunks retrieved, how many were actually relevant?

These metrics tell you if your retriever is surfacing the right context consistently.

#### LLM as Judge
Instead of manually scoring answers, you can use a strong LLM to evaluate outputs. Feed the `reference answer` and the `generated answer` to the judge model. Ask it to score on `accuracy`, `completeness`, `relevance`, and `faithfulness`. This scales evaluation across hundreds of test cases without human reviewers.

#### Golden Data (Curated Test Set):

This is your ground truth: A set of example questions with known correct answers and identified relevant context chunks. Used to benchmark both retrieval and generation. Without golden data, metrics are meaningless — you need a reference to compare against.

#### Summary

- Golden Data provides the baseline (questions + correct answers + relevant chunks).
- Retrieval Metrics measure if the retriever is surfacing the right context.
- LLM as Judge measures if the final answers are correct, complete, and useful.

![evaluation ](image-13.png)

![RAG Evaluations work flow](image-14.png)